In [1]:
# ============================================================
# CELL 1 — SETUP, LOAD DATA & BUILD BASE FEATURES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# LOAD ALL DATA
# ============================================================
results          = pd.read_csv('../data/results.csv')
races            = pd.read_csv('../data/races.csv')
drivers          = pd.read_csv('../data/drivers.csv')
constructors     = pd.read_csv('../data/constructors.csv')
pit_stops        = pd.read_csv('../data/pit_stops.csv')
qualifying       = pd.read_csv('../data/qualifying.csv')
driver_standings = pd.read_csv('../data/driver_standings.csv')
circuits         = pd.read_csv('../data/circuits.csv')

print("✅ All CSVs loaded.")

# ============================================================
# CLEAN DATA
# ============================================================
results.replace('\\N', np.nan, inplace=True)
qualifying.replace('\\N', np.nan, inplace=True)
driver_standings.replace('\\N', np.nan, inplace=True)

results['positionOrder'] = pd.to_numeric(results['positionOrder'], errors='coerce')
results['points']        = pd.to_numeric(results['points'],        errors='coerce')
results['grid']          = pd.to_numeric(results['grid'],          errors='coerce')
results['laps']          = pd.to_numeric(results['laps'],          errors='coerce')
results['position']      = pd.to_numeric(results['position'],      errors='coerce')

print("✅ Data cleaned.")

# ============================================================
# FILTER TO 2021-2024
# ============================================================
modern_races   = races[races['year'].between(2021, 2024)].copy()
modern_raceIds = modern_races['raceId'].tolist()
modern_results = results[results['raceId'].isin(modern_raceIds)].copy()

# Merge race info into results
modern_results = modern_results.merge(
    modern_races[['raceId', 'year', 'round', 'circuitId', 'name']],
    on='raceId', how='left'
)

print(f"\nDataset coverage:")
print(f"  Races   : {len(modern_races)}")
print(f"  Results : {len(modern_results)}")
print(f"  Years   : {sorted(modern_results['year'].unique())}")

# ============================================================
# TARGET VARIABLE
# ============================================================
modern_results['is_winner'] = (
    modern_results['positionOrder'] == 1
).astype(int)

total        = len(modern_results)
winners      = modern_results['is_winner'].sum()
non_winners  = total - winners
win_pct      = (winners / total) * 100

print(f"\nTarget variable — is_winner:")
print(f"  Winners (1)     : {winners}  ({win_pct:.1f}%)")
print(f"  Non-winners (0) : {non_winners}  ({100-win_pct:.1f}%)")
print(f"  Base rate       : 1 winner per race = ~5% positive class")

# ============================================================
# BASE FEATURE 1 — GRID POSITION
# ============================================================
modern_results['grid'] = modern_results['grid'].replace(0, 20)
modern_results['grid'] = modern_results['grid'].fillna(20)

# ============================================================
# BASE FEATURE 2 — DRIVER WIN RATE AT CIRCUIT
# ============================================================
circuit_wins = modern_results.groupby(
    ['driverId', 'circuitId']
).agg(
    races_at_circuit = ('raceId', 'count'),
    wins_at_circuit  = ('is_winner', 'sum')
).reset_index()

circuit_wins['win_rate_at_circuit'] = (
    circuit_wins['wins_at_circuit'] /
    circuit_wins['races_at_circuit']
)

modern_results = modern_results.merge(
    circuit_wins[['driverId', 'circuitId', 'win_rate_at_circuit']],
    on=['driverId', 'circuitId'], how='left'
)
modern_results['win_rate_at_circuit'] = (
    modern_results['win_rate_at_circuit'].fillna(0)
)

# ============================================================
# BASE FEATURE 3 — CONSTRUCTOR AVERAGE POINTS PER RACE
# ============================================================
constructor_avg = (
    modern_results.groupby('constructorId')['points']
    .mean()
    .reset_index()
)
constructor_avg.columns = ['constructorId', 'constructor_avg_points']

modern_results = modern_results.merge(
    constructor_avg, on='constructorId', how='left'
)
modern_results['constructor_avg_points'] = (
    modern_results['constructor_avg_points'].fillna(0)
)

# ============================================================
# BASE FEATURE 4 — DRIVER AVERAGE POINTS PER RACE
# ============================================================
driver_avg = (
    modern_results.groupby('driverId')['points']
    .mean()
    .reset_index()
)
driver_avg.columns = ['driverId', 'driver_avg_points']

modern_results = modern_results.merge(
    driver_avg, on='driverId', how='left'
)
modern_results['driver_avg_points'] = (
    modern_results['driver_avg_points'].fillna(0)
)

# ============================================================
# SUMMARY
# ============================================================
print(f"\nBase features built:")
print(f"  grid                    : qualifying / grid position")
print(f"  win_rate_at_circuit     : driver historical win % at this circuit")
print(f"  constructor_avg_points  : team average points per race")
print(f"  driver_avg_points       : driver average points per race")
print(f"\nFinal dataset shape : {modern_results.shape}")
print(f"\n✅ Cell 1 complete - Base features ready.")

✅ All CSVs loaded.
✅ Data cleaned.

Dataset coverage:
  Races   : 90
  Results : 1799
  Years   : [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Target variable — is_winner:
  Winners (1)     : 90  (5.0%)
  Non-winners (0) : 1709  (95.0%)
  Base rate       : 1 winner per race = ~5% positive class

Base features built:
  grid                    : qualifying / grid position
  win_rate_at_circuit     : driver historical win % at this circuit
  constructor_avg_points  : team average points per race
  driver_avg_points       : driver average points per race

Final dataset shape : (1799, 26)

✅ Cell 1 complete - Base features ready.
